# TruthGuard — Notebook 3/5
## Étape 3 : Prétraitement NLP, Split train/test & Export

> **Prérequis** : `df_raw` disponible en mémoire (issu du Notebook 1).

## 9. Prétraitement — Pipeline NLP unifié

**Étapes :**
1. Suppression des doublons et NaN
2. Normalisation (minuscules, URLs, HTML, ponctuation)
3. Tokenisation
4. Filtrage stopwords + tokens trop courts
5. Lemmatisation guidée par POS tags
6. Reconstruction du texte

In [ ]:
df = df_raw.copy()

n_avant = len(df)
df.drop_duplicates(subset="statement", inplace=True)
df.dropna(subset=["statement", "label"], inplace=True)
df = df[df["word_count"] >= 5].copy()
df.reset_index(drop=True, inplace=True)

print(f"Lignes avant nettoyage structurel : {n_avant}")
print(f"Lignes après nettoyage structurel : {len(df)}")
print(f"Supprimées : {n_avant - len(df)}")

In [ ]:
STOP_WORDS  = set(stopwords.words("english"))
LEMMATIZER  = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag: str) -> str:
    """Mappe les tags Penn Treebank vers les catégories WordNet pour la lemmatisation."""
    from nltk.corpus import wordnet
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def clean_text(text: str, use_pos_lemma: bool = True) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]

    if use_pos_lemma:
        try:
            pos_tags  = pos_tag(tokens)
            tokens = [
                LEMMATIZER.lemmatize(word, get_wordnet_pos(tag))
                for word, tag in pos_tags
            ]
        except LookupError:
            tokens = [LEMMATIZER.lemmatize(t) for t in tokens]
    else:
        tokens = [LEMMATIZER.lemmatize(t) for t in tokens]

    return " ".join(tokens)

print(" Application du nettoyage sur tout le dataset (peut prendre quelques minutes)...")
df["clean"] = df["statement"].apply(clean_text)
print(f" Nettoyage terminé. {len(df)} articles traités.")

print("\n── Exemple avant/après ──")
for i in range(2):
    print(f"\n[{i}] AVANT : {df['statement'].iloc[i][:150]}")
    print(f"[{i}] APRÈS : {df['clean'].iloc[i][:150]}")

## 10. Prétraitement — Validation & diagnostics post-nettoyage

In [ ]:
df["clean_word_count"] = df["clean"].str.split().str.len()
df["clean_char_count"] = df["clean"].str.len()
df["vocab_richness"]   = df["clean"].apply(
    lambda x: len(set(x.split())) / max(len(x.split()), 1)
)

print("── Statistiques post-nettoyage ──")
print(df.groupby("label")[["clean_word_count", "vocab_richness"]]
      .agg(["mean", "median"]).round(3))

avg_before = df["word_count"].mean()
avg_after  = df["clean_word_count"].mean()
print(f"\nMots moyens avant nettoyage : {avg_before:.1f}")
print(f"Mots moyens après nettoyage : {avg_after:.1f}")
print(f"Taux de compression         : {(1 - avg_after/avg_before)*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for label, name in [("word_count", "Avant"), ("clean_word_count", "Après")]:
    axes[0].hist(df[label].clip(upper=1500), bins=40, alpha=0.6, label=name, density=True)
axes[0].set_title("Distribution nb mots : avant vs après", fontweight="bold")
axes[0].set_xlabel("Nombre de mots")
axes[0].legend()

for lbl, color, name in [(0, "#e74c3c", "Fake"), (1, "#2980b9", "Real")]:
    axes[1].hist(df[df["label"] == lbl]["vocab_richness"], bins=30,
                 alpha=0.6, color=color, label=name, density=True)
axes[1].set_title("Richesse du vocabulaire (Type-Token Ratio)", fontweight="bold")
axes[1].set_xlabel("TTR")
axes[1].legend()

empty_after = (df["clean"].str.strip() == "").sum()
too_short   = (df["clean_word_count"] < 3).sum()
bar_data = {"Textes vides": empty_after, "Trop courts (<3 mots)": too_short,
            "OK": len(df) - empty_after - too_short}
axes[2].bar(bar_data.keys(), bar_data.values(),
            color=["#e74c3c", "#f39c12", "#27ae60"])
axes[2].set_title("Qualité après nettoyage", fontweight="bold")
axes[2].set_ylabel("Nombre d'articles")

plt.suptitle("Diagnostics post-nettoyage", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n  Textes OK après nettoyage  : {bar_data['OK']:,}")
print(f"  Textes vides après nettoyage : {empty_after}")

In [ ]:
df = df[df["clean_word_count"] >= 3].copy()
df.reset_index(drop=True, inplace=True)
print(f"Dataset final après filtrage post-nettoyage : {len(df):,} articles")
print(df["label"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, lbl, title, cmap in zip(
    axes,
    [0, 1],
    ["Fake News (texte nettoyé)", "Real News (texte nettoyé)"],
    ["Reds", "Blues"]
):
    text = " ".join(df[df["label"] == lbl]["clean"].fillna(""))
    wc = WordCloud(width=700, height=380, background_color="white",
                   colormap=cmap, max_words=80).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.axis("off")

plt.suptitle("Word Clouds — Textes nettoyés", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 11. Split train/test stratifié

In [ ]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test, idx_train, idx_test = train_test_split(
    df["clean"],
    df["label"],
    df.index,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print(f"Train : {len(X_train_text):,} articles")
print(f"Test  : {len(X_test_text):,} articles")
print(f"\nDistribution train — Fake: {(y_train==0).sum()} | Real: {(y_train==1).sum()}")
print(f"Distribution test  — Fake: {(y_test==0).sum()}  | Real: {(y_test==1).sum()}")

ratio_train = (y_train == 0).sum() / len(y_train)
ratio_test  = (y_test  == 0).sum() / len(y_test)
print(f"\nRatio Fake — train: {ratio_train:.3f} | test: {ratio_test:.3f} ")

In [ ]:
train_set = set(X_train_text.values)
test_set  = set(X_test_text.values)
overlap   = train_set & test_set
print(f"Chevauchement train/test (data leakage) : {len(overlap)} articles")
if len(overlap) == 0:
    print(" Aucun data leakage détecté.")
else:
    print("  Data leakage détecté ! Vérifier le pipeline de déduplication.")

## 12. Export du dataset prétraité

In [ ]:
df_export = df[["statement", "clean", "label", "source_df",
                "char_count", "word_count", "clean_word_count",
                "sent_count", "vocab_richness"]].copy()

df_export["split"] = "train"
df_export.loc[idx_test, "split"] = "test"

out_path = OUT_DIR / "truthguard_preprocessed.csv"
df_export.to_csv(out_path, index=False)

print(f" Dataset exporté : {out_path}")
print(f"   Dimensions     : {df_export.shape}")
print(f"   Colonnes       : {list(df_export.columns)}")
print(f"   Split train    : {(df_export['split'] == 'train').sum():,}")
print(f"   Split test     : {(df_export['split'] == 'test').sum():,}")
df_export.head(10)

In [ ]:
print("═" * 60)
print(" RÉCAPITULATIF FINAL PRÉTRAITEMENT")
print("═" * 60)
print(f"\n  Dataset brut initial  : {n_avant:,} articles")
print(f"  Après nettoyage NLP   : {len(df):,} articles")
print(f"\n  Train                 : {len(X_train_text):,}")
print(f"  Test                  : {len(X_test_text):,}")
print(f"\n  Prochaine étape       : vectorisation TF-IDF + embeddings")
print("="*60)